In [0]:
%pip install sentence-transformers faiss-cpu transformers accelerate scikit-learn
dbutils.library.restartPython()

In [0]:
from typing import List, Dict, TypedDict, Optional
from pathlib import Path
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
from transformers import pipeline

In [0]:
dbutils.library.restartPython()

In [0]:

CHUNKS_DELTA_PATH = "workspace.default.legal_docs_all_chunks"
VECTOR_INDEX_DIR = "workspace.default.legal_docs_vector_index"
EMBED_MODEL_NAME = "databricks-bge-large-en"
GENERATOR_MODEL_NAME = "google/flan-t5-base"

In [0]:
chunks_df = spark.table(CHUNKS_DELTA_PATH).toPandas()

print("Loaded chunks:", len(chunks_df))
chunks_df.head()

In [0]:
embedder = SentenceTransformer(EMBED_MODEL_NAME)
generator = pipeline(
    "text2text-generation",
    model=GENERATOR_MODEL_NAME,
    device=-1
)

In [0]:
faiss_index_path = str(Path(VECTOR_INDEX_DIR) / "legal.faiss")
index = faiss.read_index(faiss_index_path)

print("FAISS index loaded from:", faiss_index_path)

In [0]:
class DeltaFAISSRetriever:
    def __init__(self, df: pd.DataFrame, faiss_index, embed_model: SentenceTransformer):
        self.df = df.reset_index(drop=True).copy()
        self.index = faiss_index
        self.embed_model = embed_model

    def search(self, query: str, domain: Optional[str] = None, k: int = 5, fetch_k: int = 20) -> List[Dict]:
        q_emb = self.embed_model.encode(
            [f"query: {query}"],
            normalize_embeddings=True
        ).astype("float32")

        fetch_k = max(fetch_k, k)
        scores, ids = self.index.search(q_emb, fetch_k)

        results = []
        for score, idx in zip(scores[0], ids[0]):
            if idx < 0 or idx >= len(self.df):
                continue

            row = self.df.iloc[int(idx)].to_dict()

            if domain is not None and str(row.get("domain", "")).lower() != domain.lower():
                continue

            row["score"] = float(score)
            results.append(row)

            if len(results) >= k:
                break

        return results

In [0]:
retriever = DeltaFAISSRetriever(chunks_df, index, embedder)

print("Retriever ready.")

In [0]:
def build_context_block(chunks: List[Dict]) -> str:
    blocks = []
    for i, c in enumerate(chunks, start=1):
        header = (
            f"[{i}] {c.get('title', 'Unknown')} | "
            f"page {c.get('page', 'NA')} | "
            f"{c.get('source_file', 'NA')}"
        )
        body = c.get("text", "")
        blocks.append(f"{header}\n{body}")
    return "\n\n".join(blocks)